# Subgroup discovery

Subgroup discovery finds **descriptions of interesting parts of the data**, e.g.
"married people with a bachelor's degree", whose target distribution differs
from the whole population. Unlike a classifier, it reports the best individual
patterns, ranked by a quality function that trades off size against deviation.

`pysubgroup` provides

- **targets** (binary, numeric, model-based, target-free), fitted on the data,
- **quality functions** with optimistic estimates for pruning,
- **filters** (constraints), split into anti-monotone and other filters,
- exhaustive **search algorithms** (DFS, best-first, OPUS, Apriori, SD-Map*, VLSD) and beam search.

This notebook walks through them on the **Adult** census dataset from OpenML.

In [1]:
from sklearn.datasets import fetch_openml

# Adult / Census Income (OpenML dataset 1590): 48,842 people, income above 50K or not.
adult = fetch_openml("adult", version=2, as_frame=True)
# fnlwgt is a census sampling weight, not a property of the person.
data = adult.frame.drop(columns=["fnlwgt"])
features = data.drop(columns="class")
data.head()

,age,workclass,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,25,Private,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,NaN,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K


## Search space

A subgroup is a conjunction of conditions. `generate_conditions` creates the
candidates: one equality condition per category, five quantile bins per
numeric feature, and "is missing" conditions where values are missing. Searches
use at most one condition per feature.

In [2]:
from pysubgroup import generate_conditions

conditions = generate_conditions(features, n_bins=5)
print(len(conditions), "conditions")
conditions[:6]

127 conditions
[Condition(feature='age', operator='interval', value=None, lower=None, upper=np.int64(31)), Condition(feature='age', operator='interval', value=None, lower=np.int64(31), upper=np.int64(46)), Condition(feature='age', operator='interval', value=None, lower=np.int64(46), upper=np.int64(61)), Condition(feature='age', operator='interval', value=None, lower=np.int64(61), upper=np.int64(76)), Condition(feature='age', operator='interval', value=None, lower=np.int64(76), upper=None), Condition(feature='workclass', operator='eq', value='Private', lower=None, upper=None)]


## A first search: who earns more than 50K?

The target is fitted on the data and stores per-row information (here: which
rows are positive). The task combines data, target, search space, and quality
function. **WRAcc** (weighted relative accuracy) rewards subgroups that are
both large and have a much higher share of `>50K` than the 24% overall.

In [ ]:
import numpy as np
import pandas as pd

import pysubgroup as sd

target = sd.BinaryTarget().fit(data, "class", ">50K")
print(f"overall target share: {target.n_positives_ / target.n_samples_:.3f}")

task = sd.SubgroupDiscoveryTask(
    data, target, conditions, sd.WRAccQF(), result_set_size=10, depth=2
)
results = sd.DFS().execute(task)
sd.to_dataframe(results, data, target).round(4)

`to_dataframe` turns search results, i.e. `(quality, Rule)` pairs, into a table.
By default it shows each subgroup's size and the target's main statistics; the
`statistics` argument selects others by name, and `measures` adds the values of
further quality functions. Which statistics exist depends on the target:

In [ ]:
print(sd.available_statistics(target))
sd.to_dataframe(
    results[:5], data, target,
    statistics=["size", "positives", "target_share", "lift", "odds_ratio"],
    measures={"chi-squared": sd.ChiSquaredQF(), "binomial": sd.SimpleBinomialQF()},
).round(4)

## Selecting the rows of a subgroup

Each result is a `(quality, Rule)` pair. `subgroup_data` returns the rows of
`data` that one subgroup covers, as a DataFrame with the original index, for
any further analysis with pandas. Here we take the best subgroup of the first
search:

In [ ]:
quality, rule = results[0]
print(rule)
members = sd.subgroup_data(rule, data)
print(f"{len(members)} rows, share of >50K: {(members['class'] == '>50K').mean():.3f}")
members.head()

Any rule works, also one written by hand. Conditions on numeric features are
half-open intervals `(lower, upper]`; rows with a missing value never match a
condition other than "is missing". The Boolean row mask of several subgroups
at once is available from `sd.utils.subgroup_masks(results, data)`.

In [ ]:
from pysubgroup import Condition, Rule

rule = Rule((
    Condition("age", "interval", lower=30, upper=40),
    Condition("education", "eq", "Bachelors"),
))
members = sd.subgroup_data(rule, data)
members[["age", "education", "class"]].describe(include="all")

## Quality functions: size versus deviation

`StandardQF(a)` computes `(size/N)**a * (share_sg - share_dataset)`. The
exponent `a` sets the trade-off: `a=0` (`LiftQF`) ignores size and finds small,
extreme subgroups; `a=0.5` (`SimpleBinomialQF`) and `a=1` (`WRAccQF`) prefer
larger ones. The chi-squared test is another option. A minimum size keeps the
small-subgroup measures meaningful (see filters below).

In [ ]:
quality_functions = {
    "Lift (a=0)": sd.LiftQF(),
    "binomial (a=0.5)": sd.SimpleBinomialQF(),
    "WRAcc (a=1)": sd.WRAccQF(),
    "chi-squared": sd.ChiSquaredQF(),
}
tables = []
for name, qf in quality_functions.items():
    task = sd.SubgroupDiscoveryTask(
        data, target, conditions, qf, result_set_size=3, depth=2,
        filters=[sd.MinSizeFilter(200)],
    )
    found = sd.DFS().execute(task)
    table = sd.to_dataframe(found, data, target, ["size", "target_share"]).round(4)
    tables.append(table.assign(measure=name))
pd.concat(tables).set_index(["measure", "subgroup"])

## Search algorithms

All exhaustive algorithms return the same best subgroups; they differ in how
they traverse and prune the search space with optimistic estimates. **Beam
search** is a heuristic that refines only the best `beam_width` candidates per
level. Here, all find the same top 10 qualities for depth 3.

In [6]:
import time

task = sd.SubgroupDiscoveryTask(data, target, conditions, sd.WRAccQF(), 10, depth=3)
algorithms = {
    "DFS": sd.DFS(),
    "best-first": sd.BestFirstSearch(),
    "OPUS": sd.OPUS(),
    "Apriori": sd.Apriori(),
    "SD-Map*": sd.SDMapStar(),
    "VLSD": sd.VLSD(),
    "beam search (width 20)": sd.BeamSearch(beam_width=20),
}
reference = None
rows = []
for name, algorithm in algorithms.items():
    started = time.perf_counter()
    found = algorithm.execute(task)
    qualities = [quality for quality, _ in found]
    reference = reference or qualities
    rows.append({
        "algorithm": name,
        "seconds": time.perf_counter() - started,
        "best quality": qualities[0],
        "same top 10 qualities as DFS": np.allclose(qualities, reference),
    })
pd.DataFrame(rows).round(4)

,algorithm,seconds,best quality,same top 10 qualities as DFS
0,DFS,0.0870,0.0948,True
1,best-first,0.0590,0.0948,True
2,OPUS,0.0493,0.0948,True
3,Apriori,0.0504,0.0948,True
4,SD-Map*,0.6777,0.0948,True
5,VLSD,0.0653,0.0948,True
6,beam search (width 20),0.0653,0.0948,True


## Filters

Filters restrict which subgroups are reported. **Anti-monotone** filters hold for
all generalizations of a subgroup that satisfies them: once a subgroup fails,
all its refinements fail too, so the search prunes the whole branch (e.g.
a minimum size). Other filters only decide whether a subgroup enters the
results; its refinements are still searched (e.g. a minimum target share,
which a refinement can reach even if its parent does not).

Here `MinSizeFilter(0.05)` requires at least 5% of the data (a float is a
fraction of the dataset, an int a number of rows) and `MinTargetShareFilter(0.5)`
a target share of at least 50%.

In [ ]:
task = sd.SubgroupDiscoveryTask(
    data, target, conditions, sd.WRAccQF(), result_set_size=10, depth=3,
    filters=[sd.MinSizeFilter(0.05), sd.MinTargetShareFilter(0.5)],
)
print(f"minimum size: {0.05 * len(data):.0f} rows")
sd.to_dataframe(sd.DFS().execute(task), data, target, ["size", "target_share"]).round(4)

Next: `02_numeric_target.ipynb` searches for subgroups with an unusual mean of a
numeric attribute, `03_visualizing_subgroups.ipynb` shows the visualizations
proposed in the subgroup discovery literature, `04_subroc.ipynb` uses subgroup
discovery to find where a classifier performs poorly, and
`05_exceptional_model_mining.ipynb` searches for subgroups whose model (e.g. a
correlation or regression) differs from the rest of the data.